# wz-agent 交互教程

## 项目简介

wz-agent 是一个从零手搓的通用编码助手，基于 DeepSeek API，采用 ReAct（Reasoning + Acting）范式。
本 Notebook 带你逐步体验核心流程：API 调用 → ReAct 循环 → 工具调用。

## 作者信息
- 姓名: Wz9899
- GitHub: @Wz9899
- 日期: 2025-07-30

# ========================================
# 第1部分：环境配置
# ========================================

In [ ]:
# 安装依赖（如已安装可跳过）
!pip install -q openai python-dotenv click rich

In [ ]:
# 导入必要的库
import os
import sys
from dotenv import load_dotenv
from openai import OpenAI
from rich.console import Console
from rich.panel import Panel

# 加载环境变量
load_dotenv()

console = Console()

# 检查 API Key
api_key = os.environ.get("DEEPSEEK_API_KEY")
if api_key:
    console.print("[green]✅ DEEPSEEK_API_KEY 已设置[/]")
else:
    console.print("[red]❌ 请设置 DEEPSEEK_API_KEY 环境变量[/]")
    console.print("  export DEEPSEEK_API_KEY='sk-你的key'")

# ========================================
# 第2部分：工具定义
# ========================================

In [ ]:
import json
from typing import Any


class ReadTool:
    """读取文件内容的工具"""

    name = "read"
    description = "读取指定路径的文件内容"

    def to_openai_function(self) -> dict:
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": {
                    "type": "object",
                    "properties": {
                        "path": {
                            "type": "string",
                            "description": "文件路径（相对或绝对）"
                        }
                    },
                    "required": ["path"]
                }
            }
        }

    def run(self, path: str) -> str:
        try:
            with open(path, "r", encoding="utf-8") as f:
                content = f.read()
            # 截断过长内容
            if len(content) > 3000:
                content = content[:3000] + "\n... (内容已截断)"
            return content
        except FileNotFoundError:
            return f"错误：文件不存在 —— {path}"
        except Exception as e:
            return f"错误：{e}"


class WriteTool:
    """创建或覆盖文件的工具"""

    name = "write"
    description = "创建新文件或覆盖已有文件，会自动创建父目录"

    def to_openai_function(self) -> dict:
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": {
                    "type": "object",
                    "properties": {
                        "path": {
                            "type": "string",
                            "description": "文件路径"
                        },
                        "content": {
                            "type": "string",
                            "description": "要写入的内容"
                        }
                    },
                    "required": ["path", "content"]
                }
            }
        }

    def run(self, path: str, content: str) -> str:
        try:
            os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
            with open(path, "w", encoding="utf-8") as f:
                f.write(content)
            return f"✅ 文件已写入：{path}（{len(content)} 字符）"
        except Exception as e:
            return f"错误：{e}"


class BashTool:
    """执行 Shell 命令的工具"""

    name = "bash"
    description = "执行 shell 命令并返回输出结果"

    def to_openai_function(self) -> dict:
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": {
                    "type": "object",
                    "properties": {
                        "command": {
                            "type": "string",
                            "description": "要执行的 shell 命令"
                        }
                    },
                    "required": ["command"]
                }
            }
        }

    def run(self, command: str) -> str:
        import subprocess
        try:
            result = subprocess.run(
                command, shell=True, capture_output=True, text=True, timeout=30
            )
            output = result.stdout
            if result.stderr:
                output += "\n[stderr]\n" + result.stderr
            return output if output.strip() else "(命令执行完毕，无输出)"
        except subprocess.TimeoutExpired:
            return "错误：命令执行超时（30 秒）"
        except Exception as e:
            return f"错误：{e}"


# 工具注册表
ALL_TOOLS: dict[str, Any] = {
    "read": ReadTool(),
    "write": WriteTool(),
    "bash": BashTool(),
}

console.print("[green]✅ 工具已注册:[/]", ", ".join(ALL_TOOLS.keys()))

# ========================================
# 第3部分：智能体构建
# ========================================

In [ ]:
import json


class CodingAgent:
    """ReAct 编码智能体"""

    def __init__(self, api_key: str, model: str = "deepseek-chat"):
        self.client = OpenAI(
            api_key=api_key,
            base_url="https://api.deepseek.com",
        )
        self.model = model
        self.system_prompt = """你是一个编码助手，可以用中文回答。
你有以下工具可用：
- read: 读取文件内容
- write: 创建或覆盖文件
- bash: 执行 shell 命令

当需要操作文件或执行命令时，请使用工具。完成任务后，总结你做了什么。"""

    def run(self, user_message: str, max_steps: int = 10) -> str:
        """
        ReAct 循环：思考 → 行动 → 观察 → 再思考
        """
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": user_message},
        ]

        # 构建工具 schema
        tool_schemas = [t.to_openai_function() for t in ALL_TOOLS.values()]

        for step in range(max_steps):
            console.print(f"\n[bold yellow]--- 第 {step + 1} 步 ---[/]")

            # 调用 LLM
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                tools=tool_schemas,
            )

            msg = response.choices[0].message

            # 如果 LLM 想调用工具
            if msg.tool_calls:
                # 把 assistant 消息加入对话
                messages.append(msg.model_dump())

                for tool_call in msg.tool_calls:
                    tool_name = tool_call.function.name
                    tool_args = json.loads(tool_call.function.arguments)

                    console.print(
                        f"  🔧 调用工具: [cyan]{tool_name}[/] {tool_args}"
                    )

                    # 执行工具
                    tool = ALL_TOOLS.get(tool_name)
                    if tool:
                        result = tool.run(**tool_args)
                    else:
                        result = f"未知工具：{tool_name}"

                    # 截断过长结果
                    result_display = result[:500] + "..." if len(result) > 500 else result
                    console.print(f"  📋 结果: {result_display}")

                    # 把工具结果加入对话
                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "content": result,
                    })

            else:
                # LLM 直接回复，循环结束
                final_reply = msg.content
                console.print(f"\n[bold green]✅ Agent 完成：[/]\n{final_reply}")
                return final_reply

        return "⚠️ 达到最大步数限制，Agent 未完成任务。"


# 创建 agent 实例
agent = CodingAgent(api_key=os.environ["DEEPSEEK_API_KEY"])
console.print("[green]✅ Agent 初始化完成[/]")

# ========================================
# 第4部分：基础功能演示
# ========================================

In [ ]:
# 示例1：纯对话（无需工具）
print("=== 示例1：基础对话 ===")
result = agent.run("用一句话介绍什么是 ReAct 循环。")
print(result)

In [ ]:
# 示例2：文件操作
print("=== 示例2：创建文件 ===")
result = agent.run("创建一个 hello.py 文件，内容是一个打印 'Hello from wz-agent!' 的 Python 脚本")
print("\n最终结果:", result)

In [ ]:
# 验证文件是否真的创建了
!cat hello.py 2>/dev/null || type hello.py

In [ ]:
# 示例3：执行命令
print("=== 示例3：运行刚创建的脚本 ===")
result = agent.run("用 bash 运行 python hello.py 并告诉我输出结果")
print("\n最终结果:", result)

# ========================================
# 第5部分：复杂场景演示
# ========================================

In [ ]:
# 示例4：多步骤任务 —— 创建一个猜数字游戏
print("=== 示例4：创建猜数字游戏 ===")
task = """
请帮我完成以下任务：
1. 创建一个 guess_number.py 文件，实现猜数字游戏（1-100 随机数，最多 7 次机会）
2. 运行 python guess_number.py 验证语法没问题
3. 用 read 工具读取 guess_number.py 内容，确认代码完整
"""
result = agent.run(task)
print("\n最终结果:", result)

In [ ]:
# 验证生成的文件
!cat guess_number.py 2>/dev/null || type guess_number.py

# ========================================
# 第6部分：清理临时文件
# ========================================

In [ ]:
# 清理演示过程中创建的文件
import os
for f in ["hello.py", "guess_number.py"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"🗑️ 已删除：{f}")
print("✅ 清理完成")

# ========================================
# 第7部分：总结与展望
# ========================================

## 项目总结

### 实现的功能
- ✅ DeepSeek API 连通（OpenAI 兼容接口）
- ✅ ReAct 循环 v1：单轮 API 调用
- ✅ ReAct 循环 v2（[]）：工具调用 + 消息循环
- ✅ 三个基础工具：read / write / bash（）
- ✅ 工具基类：自动从 run() 签名推断参数 schema
- ✅ 多步骤任务编排

### 核心技术点
- **ReAct 范式**：思考 → 行动 → 观察循环，LLM 自主决定何时调用工具
- **Tool Calling**：利用 OpenAI Function Calling 协议，DeepSeek 原生支持
- **消息管理**：完整的 system → user → assistant → tool 消息链
- **从零手搓**：核心循环不到 100 行，无框架依赖

### 遇到的挑战
- **工具结果截断**：LLM 上下文有限，长输出需要截断或摘要
- **循环终止条件**：需要设置 max_steps 防止无限循环，后续可加入更智能的终止判断
- **错误处理**：工具执行可能失败，需要把错误信息友好地喂回 LLM 让其自行修复

### 未来改进方向
- [ ] edit 工具：精确文本匹配替换
- [ ] Bash 安全模式（auto / plan 双模式）
- [ ] 需求澄清阶段：追问用户需求，输出 spec.md
- [ ] 错误自动重试（最多 3 次）
- [ ] CLI 入口完善（click 命令行参数解析）
- [ ] 集成 triage（issue 管理）+ to-tickets（任务拆解）